<a href="https://colab.research.google.com/github/Rohan-134v/PES2UG23CS489-GenAI/blob/main/Advanced_prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install python-dotenv --upgrade --quiet langchain langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 7.8 MB/s eta 0:00:00


In [2]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

Enter your Groq API Key: ··········


In [3]:
question = "Sarah has 4 notebooks. She buys 3 more packs of notebooks. Each pack contains 5 notebooks. How many notebooks does she have now?"

prompt_standard = f"Answer this question: {question}"
print("--- STANDARD (Llama3.1-8b) ---")
print(llm.invoke(prompt_standard).content)

--- STANDARD (Llama3.1-8b) ---
To find out how many notebooks Sarah has now, we need to add the number of notebooks she already has to the number of notebooks she buys.

Sarah already has 4 notebooks. 

She buys 3 packs of notebooks, and each pack contains 5 notebooks. So, she buys 3 x 5 = 15 notebooks.

Now, we add the number of notebooks she already has to the number of notebooks she buys: 4 + 15 = 19.

So, Sarah now has 19 notebooks.


In [4]:
prompt_cot = f"Answer this question. Let's think step by step. {question}"

print("--- Chain of Thought (Llama3.1-8b) ---")
print(llm.invoke(prompt_cot).content)

--- Chain of Thought (Llama3.1-8b) ---
To find out how many notebooks Sarah has now, we need to follow these steps:

1. Sarah already has 4 notebooks.
2. She buys 3 more packs of notebooks. Each pack contains 5 notebooks.
3. To find out how many notebooks she bought in total, we multiply the number of packs by the number of notebooks in each pack: 3 packs * 5 notebooks/pack = 15 notebooks.
4. Now, we add the number of notebooks she already had to the number of notebooks she bought: 4 notebooks + 15 notebooks = 19 notebooks.

So, Sarah now has 19 notebooks.


In [5]:
from dotenv import load_dotenv
load_dotenv()

import getpass
import os
from langchain_groq import ChatGroq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

problem = "How can I get my 5-year-old to eat vegetables?"

prompt_branch = ChatPromptTemplate.from_template(
    "Problem: {problem}. Give me one unique, creative solution. Solution {id}:"
)

branches = RunnableParallel(
    sol1=prompt_branch.partial(id="1") | llm | StrOutputParser(),
    sol2=prompt_branch.partial(id="2") | llm | StrOutputParser(),
    sol3=prompt_branch.partial(id="3") | llm | StrOutputParser(),
)

prompt_judge = ChatPromptTemplate.from_template(
    """
    I have three proposed solutions for: '{problem}'

    1: {sol1}
    2: {sol2}
    3: {sol3}

    Act as a Child Psychologist. Pick the most sustainable one (not bribery) and explain why.
    """
)

tot_chain = (
    RunnableParallel(problem=RunnableLambda(lambda x: x), branches=branches)
    | (lambda x: {**x["branches"], "problem": x["problem"]})
    | prompt_judge
    | llm
    | StrOutputParser()
)

print("--- Tree of Thoughts (ToT) Result ---")
print(tot_chain.invoke(problem))

--- Tree of Thoughts (ToT) Result ---
As a child psychologist, I would recommend Solution 2: Involve Them in the Process of Growing Their Own Vegetables. This approach is the most sustainable and effective method for encouraging a 5-year-old to eat vegetables. Here's why:

1. **Ownership and Responsibility**: By involving your child in the process of growing their own vegetables, they feel a sense of ownership and responsibility for the food they grow and eat. This fosters a positive relationship with vegetables and encourages them to take care of their garden.
2. **Understanding of Food Source**: Growing their own vegetables helps your child understand where their food comes from and the effort that goes into producing it. This can lead to a greater appreciation for the food they eat.
3. **Developmental Benefits**: Gardening and nurturing plants can have numerous developmental benefits for children, including improved fine motor skills, hand-eye coordination, and cognitive development

In [7]:
prompt_draft = ChatPromptTemplate.from_template(
    "Write a 1-sentence movie plot about: {topic}. Genre: {genre}."
)

drafts = RunnableParallel(
    draft_scifi=prompt_draft.partial(genre="Sci-Fi") | llm | StrOutputParser(),
    draft_romance=prompt_draft.partial(genre="Romance") | llm | StrOutputParser(),
    draft_horror=prompt_draft.partial(genre="Horror") | llm | StrOutputParser(),
)

prompt_combine = ChatPromptTemplate.from_template(
    """
    I have three movie ideas for the topic '{topic}':
    1. Sci-Fi: {draft_scifi}
    2. Romance: {draft_romance}
    3. Horror: {draft_horror}

    Your task: Create a new Mega-Movie that combines the TECHNOLOGY of Sci-Fi, the PASSION of Romance, and the FEAR of Horror.
    Write one paragraph.
    """
)

got_chain = (
    RunnableParallel(topic=RunnableLambda(lambda x: x), drafts=drafts)
    | (lambda x: {**x["drafts"], "topic": x["topic"]})
    | prompt_combine
    | llm
    | StrOutputParser()
)

print("--- Graph of Thoughts (GoT) Result ---")
print(got_chain.invoke("Time Travel"))

--- Graph of Thoughts (GoT) Result ---
In "Echoes of Eternity," a reclusive and brilliant physicist, Emma Taylor, stumbles upon a revolutionary time travel technology that allows her to traverse the past with the precision of a Swiss watch. Her initial intention is to use the device to revive her late fiancé, Max, who met a tragic demise in a car accident. As she journeys back in time, she becomes increasingly obsessed with the possibility of reuniting with Max, but with each attempt, she inadvertently creates a labyrinthine web of parallel timelines, each with their own version of reality. However, as Emma delves deeper into the past, she begins to realize that she's not the only one searching for Max - a malevolent entity, born from the darkest corners of human consciousness, has also discovered the time travel technology, and it will stop at nothing to claim Max's soul, threatening to erase Emma's existence from the fabric of time.
